<a href="https://colab.research.google.com/github/Ali02-r/Komputasi-Numerik/blob/main/eliminasi_gauss.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Eliminasi Gauss untuk SPL 3 Variabel

Notebook ini menyelesaikan tiga persamaan linear dengan tiga bilangan yang belum diketahui (x1, x2, x3) memakai metode **Eliminasi Gauss**. Langkahnya mengikuti dokumen algoritma di repository ini.

**Cara pakai:** jalankan cell dari atas ke bawah dengan `Shift + Enter`. Kalau data soal diubah, jalankan ulang mulai dari Langkah 1.

**Catatan indeks:** pada dokumen algoritma, baris dan kolom dihitung mulai dari 1, sedangkan Python mulai dari 0. Jadi `M[1][1]` di algoritma ditulis `M[0][0]` di kode.

**Alur program secara garis besar:**

| Langkah | Isi | Hasil |
|---------|-----|-------|
| 1 | Data soal (`a` dan `c`) | Koefisien dan konstanta siap dipakai |
| 2 | Susun matriks augmented | Tabel `M` berukuran 3 x 4 |
| 3 | Eliminasi maju | `M` berbentuk tangga, atau berhenti jika tidak ada solusi tunggal |
| 4 | Substitusi mundur | Nilai x3, x2, x1 |
| 5 | Jalankan penyelesaian | Hasil atau pesan "tidak ada solusi tunggal" |
| 6 | Cek jawaban | Bukti bahwa hasilnya cocok dengan soal |
| 7 | Uji kasus khusus | Bukti program menangani soal tanpa solusi tunggal |

Setiap cell kode diberi komentar pada tiap barisnya. Komentar diawali tanda `#` dan tidak mempengaruhi jalannya program.

## Langkah 1: Data soal

Penjelasan cell di bawah ini:

- `a` adalah *list* yang berisi tiga *list* lain. Setiap list di dalamnya mewakili satu persamaan dan berisi koefisien x1, x2, x3 secara berurutan. Misalnya `a[0]` adalah persamaan pertama, dan `a[0][1]` adalah koefisien x2 pada persamaan pertama (nilainya 2).
- `c` adalah list yang berisi konstanta di ruas kanan tiap persamaan, dengan urutan yang sama seperti `a`.
- Untuk soal lain, cukup ganti angka-angkanya. Jika suatu persamaan tidak memuat variabel tertentu, tulis koefisiennya 0.

In [ ]:
# Soal yang diselesaikan (tiga persamaan, tiga bilangan yang belum diketahui):
#   x1 + 2·x2 +   x3 = 2
#  2x1 +   x2 -   x3 = 7
#  3x1        +   x3 = 0

# 'a' adalah list yang berisi 3 list; tiap list di dalamnya adalah satu persamaan.
# Isi tiap list adalah koefisien x1, x2, x3 secara berurutan.
a = [[1, 2,  1],     # persamaan 1: 1·x1 + 2·x2 + 1·x3
     [2, 1, -1],     # persamaan 2: 2·x1 + 1·x2 - 1·x3
     [3, 0,  1]]     # persamaan 3: 3·x1 + 0·x2 + 1·x3 (koefisien x2 sama dengan 0)

# 'c' berisi konstanta di ruas kanan; urutannya sama dengan persamaan di atas.
c = [2, 7, 0]

## Langkah 2: Susun matriks augmented

**Matriks augmented** adalah satu tabel yang menggabungkan koefisien dan konstanta. Dengan satu tabel, setiap operasi pada baris otomatis berlaku untuk koefisien sekaligus konstantanya.

Penjelasan cell di bawah ini:

- `buat_augmented(a, c)` membuat tabel `M` berukuran 3 baris x 4 kolom. Tiga kolom pertama berisi koefisien dan kolom keempat berisi konstanta.
- Setiap angka diubah menjadi bilangan desimal dengan `float(...)` supaya pembagian pada langkah berikutnya tidak dibulatkan.
- `cetak(M)` hanya berfungsi menampilkan tabel dengan rapi dan tidak mengubah isinya.

In [ ]:
def buat_augmented(a, c):
    # Menggabungkan koefisien 'a' dan konstanta 'c' menjadi satu matriks berukuran 3 x 4
    M = []                                    # wadah kosong untuk menampung baris-baris matriks
    for i in range(3):                        # i = 0, 1, 2 -> baris ke-1, ke-2, ke-3
        # float(v) mengubah tiap angka menjadi desimal supaya pembagian tidak membulat.
        # '+ [float(c[i])]' menambahkan konstanta sebagai elemen ke-4 pada baris tersebut.
        baris = [float(v) for v in a[i]] + [float(c[i])]
        M.append(baris)                       # masukkan baris yang sudah jadi ke dalam M
    return M                                  # kembalikan matriks hasilnya


def cetak(M):
    # Hanya untuk menampilkan matriks dengan rapi; isi M tidak berubah
    for baris in M:                           # ambil satu baris dalam satu waktu
        # baris[:3] = tiga elemen pertama (koefisien).
        # f"{v:7.2f}" = angka ditulis selebar 7 karakter dengan 2 angka di belakang koma.
        kiri = "  ".join(f"{v:7.2f}" for v in baris[:3])
        # baris[3] = elemen keempat (konstanta), dipisah garis tegak '|' agar mudah dibaca
        print(f"[ {kiri} | {baris[3]:7.2f} ]")


M = buat_augmented(a, c)                      # bentuk matriks augmented dari data soal
print("Matriks augmented awal:")              # judul sebelum matriks ditampilkan
cetak(M)                                      # tampilkan matriksnya

## Langkah 3: Eliminasi maju

Untuk tiap kolom, pivot diperiksa lebih dulu. Jika pivot bernilai 0, program mencari baris di bawahnya yang elemen kolomnya tidak 0 lalu menukarnya. Jika tidak ada baris yang bisa ditukar, fungsi mengembalikan `False` karena sistem tidak punya solusi tunggal.

Setelah pivot aman, elemen di bawahnya dinolkan: hitung `faktor = M[i][k] / M[k][k]`, lalu kurangkan baris `i` dengan faktor kali baris `k`. Fungsi ini mengubah `M` secara langsung.

In [ ]:
def eliminasi_maju(M):
    # Mengembalikan True jika berhasil, False jika tidak ada solusi tunggal
    for k in range(3):                        # kolom 1, 2, 3 (indeks 0, 1, 2)
        # --- periksa pivot ---
        if abs(M[k][k]) < 1e-12:              # pivot dianggap 0
            baris_tukar = None
            for i in range(k + 1, 3):         # cari baris di bawah yang tidak 0
                if abs(M[i][k]) > 1e-12:
                    baris_tukar = i
                    break
            if baris_tukar is None:
                return False                  # tidak ada baris yang bisa ditukar
            M[k], M[baris_tukar] = M[baris_tukar], M[k]
            print(f"Baris {k + 1} ditukar dengan baris {baris_tukar + 1}")

        # --- nolkan elemen di bawah pivot ---
        for i in range(k + 1, 3):
            faktor = M[i][k] / M[k][k]
            for j in range(k, 4):
                M[i][j] = M[i][j] - faktor * M[k][j]

        print(f"Setelah kolom {k + 1} diproses:")
        cetak(M)

    return True

## Langkah 4: Substitusi mundur

Setelah `M` berbentuk tangga, x3 dihitung dari baris terakhir. Nilai x3 dipakai untuk mencari x2, lalu x2 dan x3 dipakai untuk mencari x1.

In [ ]:
def substitusi_mundur(M):
    x3 = M[2][3] / M[2][2]
    x2 = (M[1][3] - M[1][2] * x3) / M[1][1]
    x1 = (M[0][3] - M[0][1] * x2 - M[0][2] * x3) / M[0][0]
    return x1, x2, x3

## Langkah 5: Jalankan penyelesaian

Eliminasi maju dijalankan lebih dulu. Substitusi mundur hanya dilakukan kalau eliminasi berhasil; jika tidak, program menampilkan pesan bahwa sistem tidak punya solusi tunggal.

In [ ]:
ada_solusi = eliminasi_maju(M)

if ada_solusi:
    x1, x2, x3 = substitusi_mundur(M)
    print()
    print("Hasil:")
    print(f"x1 = {x1:.4f}")
    print(f"x2 = {x2:.4f}")
    print(f"x3 = {x3:.4f}")
else:
    print("Tidak ada solusi tunggal")

## Langkah 6: Cek jawaban

Nilai x1, x2, x3 dimasukkan kembali ke tiap persamaan. Jika ruas kiri sama dengan ruas kanan, jawabannya benar.

In [ ]:
if ada_solusi:
    for i in range(3):
        ruas_kiri = a[i][0] * x1 + a[i][1] * x2 + a[i][2] * x3
        print(f"Persamaan {i + 1}: ruas kiri = {ruas_kiri:.4f}, ruas kanan = {c[i]}")

## Langkah 7: Uji soal tanpa solusi tunggal

Persamaan kedua di bawah ini kelipatan dua dari persamaan pertama, sehingga sistemnya tidak punya solusi tunggal. Program seharusnya menukar baris lebih dulu, lalu berhenti dengan pesan yang sesuai.

In [ ]:
a_uji = [[1, 2, 1],
         [2, 4, 2],      # kelipatan dua dari persamaan 1
         [3, 0, 1]]
c_uji = [2, 4, 0]

M_uji = buat_augmented(a_uji, c_uji)

if eliminasi_maju(M_uji):
    print("Ada solusi tunggal")
else:
    print("Tidak ada solusi tunggal")